# Détection de Fraude par Carte Bancaire — Analyse Exploratoire des Données (EDA)

**Auteur :** Anas  
**Dataset :** Credit Card Fraud Detection (ULB / Kaggle)  
**Objectifs :**
1. Contrôler l'intégrité du dataset (colonnes, types, valeurs manquantes)
2. Analyser l'extrême déséquilibre de classe (0.17% de fraudes)
3. Étudier la variable `Amount` (montants, log-transformation) et `Time` (2 jours, heure de la journée)
4. Identifier les variables les plus discriminantes (V1-V28)
5. Justifier les choix méthodologiques : découpage temporel, métrique PR-AUC, matrice de coût.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.fraud.data.loader import load_raw_data

# Configuration visuelle
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 35)

## 1. Chargement et vérification des données

In [2]:
df = load_raw_data('data/raw/creditcard.csv')
print(f"Dimensions : {df.shape[0]:,} lignes, {df.shape[1]} colonnes")
print(f"Valeurs nulles : {df.isna().sum().sum()}")
df.head()

## 2. Analyse du déséquilibre de classe

In [3]:
counts = df['Class'].value_counts()
fraud_rate = counts[1] / len(df) * 100
print(f"Légitimes (0) : {counts[0]:,} ({100 - fraud_rate:.2f}%)")
print(f"Fraudes (1)   : {counts[1]:,} ({fraud_rate:.3f}%)")
print(f"Ratio : 1 fraude pour {int(counts[0]/counts[1])} transactions légitimes")

## 3. Analyse de la variable `Amount`

In [4]:
stats_amount = df.groupby('Class')['Amount'].describe()
display(stats_amount)
print(f"Montant total des fraudes : {df[df['Class'] == 1]['Amount'].sum():,.2f} €")

## 4. Analyse temporelle et heure de la journée

In [5]:
df['Hour'] = (df['Time'] // 3600) % 24
hour_summary = df.groupby(['Hour', 'Class']).size().unstack(fill_value=0)
hour_summary['Fraud_Rate_%'] = (hour_summary[1] / (hour_summary[0] + hour_summary[1])) * 100
display(hour_summary.head(10))

## 5. Corrélations avec la variable cible `Class`

In [6]:
corrs = df.corr()['Class'].drop(['Class', 'Hour']).sort_values()
top_neg = corrs.head(5)
top_pos = corrs.tail(5)
print("Top corrélations négatives :\n", top_neg)
print("\nTop corrélations positives :\n", top_pos)

## 6. Conclusions de l'analyse exploratoire

1. **Déséquilibre extrême** : 492 fraudes parmi 284 807 transactions (0.172%). L'accuracy est une métrique trompeuse. La métrique clé est **PR-AUC (Precision-Recall AUC)** ainsi que la fonction de coût métier.
2. **Distribution des montants** : Montant médian des fraudes (~9.25 €), mais montant moyen plus élevé (122.21 €) avec des fraudes allant jusqu'à 2 125 €. La transformation `log1p(Amount)` est indispensable.
3. **Distribution temporelle** : Données couvrant exactement 48h (2 jours). La feature `Hour` permet de capter la baisse d'activité nocturne où la proportion relative de fraude est plus élevée.
4. **Split temporel** : Le split doit impérativement respecter l'ordre temporel (`Time`) pour simuler les conditions réelles de production et éviter toute fuite d'information.